In [3]:
import glob, json, re
from pathlib import Path

ROOT = Path.cwd().parent                       # .../SAIT (running from notebooks/)
PROBLEM_DIR = ROOT / 'data' / 'problems'        # <- your folder name as given

pdfs = sorted(glob.glob(str(PROBLEM_DIR / '*.pdf')))
assert pdfs, f'No PDFs found in {PROBLEM_DIR} — check the folder name/path'
PDF = pdfs[0]
print(f'{len(pdfs)} problem sheet(s) found; testing on:', Path(PDF).name)

6 problem sheet(s) found; testing on: AP$.pdf


In [4]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader(PDF).load()
raw = '\n'.join(p.page_content for p in pages)
print(f'{len(pages)} pages, {len(raw)} chars\n')
print('=' * 70)
print(raw[:3000])                              # first ~1.5 pages — read the markers
print('...')

/var/folders/hw/9rk5hsh90j51sypntxw27ss00000gn/T/ipykernel_37808/2427718707.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/afrahalharbi/Desktop/Imperial/Project/code/SAIT/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


14 pages, 23652 chars

Introduction to Algorithms: 6.006 
Massachusetts Institute of Technology 
Instructors: Erik Demaine, Jason Ku, and Justin Solomon Problem Set 4 
Problem Set 4 
Please write your solutions in the L ATEX and Python templates provided. Aim for concise 
solutions; convoluted and obtuse descriptions might receive low marks, even when they are 
correct. 
Problem 4-1. [10 points] Binary Tree Practice 
(a) [2 points] The Set Binary Tree T below is not height-balanced but does satisfy the 
binary search tree property, assuming the key of each integer item is itself. Indicate 
the keys of all nodes that are not height-balanced and compute their skew. 
47 
16 
3 37 
35 
28 
84 
64 
49 
86 
88 
Solution: The nodes containing the keys 16 and 37 are not height balanced. Their 
skews are 2 and −2 respectively. 
Rubric: 
• 1 point for each correct node and skew
2 Problem Set 4 
(b) [5 points] Perform the following insertions and deletions, one after another in se-
quence on T, b

In [5]:
# Find every line that LOOKS like a section marker — this tells you the exact
# wording and consistency across the whole sheet before you write the regexes.
candidates = re.findall(
    r'^.{0,80}?(?:problem|question|exercise|solution|answer|mistake|error|hint).{0,60}$',
    raw, flags=re.IGNORECASE | re.MULTILINE
)
print(f'{len(candidates)} marker-like lines:\n')
for c in candidates:
    print('  |', c.strip()[:100])

30 marker-like lines:

  | Instructors: Erik Demaine, Jason Ku, and Justin Solomon Problem Set 4
  | Problem Set 4
  | Problem 4-1. [10 points] Binary Tree Practice
  | 2 Problem Set 4
  | Solution:
  | 3 Problem Set 4
  | delete(84) (Two solutions, swap down predecessor/successor)
  | 4 Problem Set 4
  | Solution: Node containing 16 is not height-balanced.
  | 5 Problem Set 4
  | Problem 4-2. Heap Practice [10 points]
  | Solution: Min-heap 4
  | Solution: Max-heap 701
  | Solution: Neither: three swaps sufﬁce to transform into a min-heap
  | Solution: Min-heap 1
  | 6 Problem Set 4
  | Problem 4-3. [10 points] Gardening Contest
  | Solution: Build a max-heap from array A keyed on the garden scores s
  | Solution: For this problem, we cannot afford O(nx log |A|) time to repeatedly delete
  | 7 Problem Set 4
  | Problem 4-4. [15 points] Solar Supply
  | 8 Problem Set 4
  | 9 Problem Set 4
  | Problem 4-5. [15 points] Robot Wrangling
  | 10 Problem Set 4
  | Problem 4-6. [40 points] πz2

In [6]:
# ---- EDIT THESE to match your sheet ----
PROBLEM_PAT  = r'Problem\s+(\d+[a-z]?)'        # captures the problem number
SOLUTION_PAT = r'Solution\s*[:.]'
MISTAKES_PAT = r'Common\s+mistakes?\s*[:.]'
# ----------------------------------------

def parse_sheet(raw_text):
    """Split raw text into problem records on the markers."""
    # split so each block starts at a 'Problem N' marker
    blocks = re.split(f'(?={PROBLEM_PAT})', raw_text)
    records = []
    for block in blocks:
        m = re.match(PROBLEM_PAT, block)
        if not m:
            continue                                  # preamble before Problem 1
        pid = m.group(1)

        sol_m  = re.search(SOLUTION_PAT, block)
        mist_m = re.search(MISTAKES_PAT, block)

        q_end   = sol_m.start() if sol_m else len(block)
        s_start = sol_m.end() if sol_m else None
        s_end   = mist_m.start() if mist_m else len(block)

        records.append({
            'problem_id': f'{Path(PDF).stem}-p{pid}',
            'question': block[m.end():q_end].strip(),
            'solution_text': block[s_start:s_end].strip() if s_start else '',
            'misconceptions_raw': block[mist_m.end():].strip() if mist_m else '',
        })
    return records

records = parse_sheet(raw)
print(f'parsed {len(records)} problems')
for r in records:
    flags = []
    if not r['solution_text']: flags.append('NO SOLUTION FOUND')
    if not r['misconceptions_raw']: flags.append('no mistakes section')
    print(f"  {r['problem_id']:20} q:{len(r['question']):5}ch  "
          f"sol:{len(r['solution_text']):5}ch  {' | '.join(flags)}")

parsed 6 problems
  AP$-p4               q:  330ch  sol: 2166ch  no mistakes section
  AP$-p4               q:  404ch  sol:  692ch  no mistakes section
  AP$-p4               q:  704ch  sol: 2284ch  no mistakes section
  AP$-p4               q: 1447ch  sol: 4422ch  no mistakes section
  AP$-p4               q: 1439ch  sol: 2105ch  no mistakes section
  AP$-p4               q: 1539ch  sol: 5633ch  no mistakes section


In [7]:
r = records[0]                                 # inspect each index in turn
print('PROBLEM_ID:', r['problem_id'])
print('\n--- QUESTION ---\n', r['question'][:800])
print('\n--- SOLUTION ---\n', r['solution_text'][:800])
print('\n--- MISTAKES (raw) ---\n', r['misconceptions_raw'][:600])

PROBLEM_ID: AP$-p4

--- QUESTION ---
 -1. [10 points] Binary Tree Practice 
(a) [2 points] The Set Binary Tree T below is not height-balanced but does satisfy the 
binary search tree property, assuming the key of each integer item is itself. Indicate 
the keys of all nodes that are not height-balanced and compute their skew. 
47 
16 
3 37 
35 
28 
84 
64 
49 
86 
88

--- SOLUTION ---
 The nodes containing the keys 16 and 37 are not height balanced. Their 
skews are 2 and −2 respectively. 
Rubric: 
• 1 point for each correct node and skew
2 Problem Set 4 
(b) [5 points] Perform the following insertions and deletions, one after another in se-
quence on T, by adding or removing a leaf while maintaining the binary search tree 
property (a key may need to be swapped down into a leaf). For this part, do not use 
rotations to balance the tree. Draw the modiﬁed tree after each operation. 
1 T.insert(2) 
2 T.delete(49) 
3 T.delete(35) 
4 T.insert(85) 
5 T.delete(84) 
Solution: 
47 
16 
3 37 
35

In [8]:
def split_misconceptions(raw_mistakes, pid):
    if not raw_mistakes:
        return []
    # split on bullets / dashes / numbered items — adjust if your sheet differs
    items = re.split(r'\n\s*(?:[-•*]|\d+[.)])\s*', '\n' + raw_mistakes)
    items = [i.strip() for i in items if i.strip()]
    return [
        {'id': f'{pid}-m{k+1}', 'description': item, 'remediation_hint': ''}
        for k, item in enumerate(items)
    ]

for r in records:
    r['misconceptions'] = split_misconceptions(r.pop('misconceptions_raw'), r['problem_id'])

for r in records:
    print(f"{r['problem_id']}: {len(r['misconceptions'])} misconception(s)")
    for mc in r['misconceptions']:
        print('   -', mc['description'][:90])

AP$-p4: 0 misconception(s)
AP$-p4: 0 misconception(s)
AP$-p4: 0 misconception(s)
AP$-p4: 0 misconception(s)
AP$-p4: 0 misconception(s)
AP$-p4: 0 misconception(s)


In [9]:
OUT = ROOT / 'data' / 'problems.json'
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(records, f, indent=2, ensure_ascii=False)
print(f'wrote {len(records)} problems to {OUT}')
print('\nNext: open it, review boundaries, transcribe the visual solution, add hints.')

wrote 6 problems to /Users/afrahalharbi/Desktop/Imperial/Project/code/SAIT/data/problems.json

Next: open it, review boundaries, transcribe the visual solution, add hints.
